# Lab type: review
# Course: ML201 — Applied Machine Learning
# Lesson: Random Forests in Depth
# Task: The code below is correct and working. Read each section, run it, then answer the judgment questions in the markdown cells below each block.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

np.random.seed(42)
n = 2000

income = np.random.normal(55000, 20000, n).clip(15000, 150000)
debt_ratio = np.random.beta(2, 5, n)
credit_score = np.random.normal(670, 80, n).clip(300, 850)
employment_years = np.random.exponential(5, n).clip(0, 40)
num_accounts = np.random.randint(1, 15, n)
region = np.random.randint(0, 5, n)
account_type = np.random.randint(0, 3, n)

log_odds = (
    -3.0
    + 1.5 * debt_ratio
    - 0.003 * (credit_score - 670) / 80
    - 0.00001 * income
    - 0.05 * employment_years
    + 0.02 * num_accounts
)
prob_default = 1 / (1 + np.exp(-log_odds))
target = (np.random.rand(n) < prob_default).astype(int)

feature_names = ['income', 'debt_ratio', 'credit_score', 'employment_years',
                 'num_accounts', 'region', 'account_type']
X = pd.DataFrame({
    'income': income,
    'debt_ratio': debt_ratio,
    'credit_score': credit_score,
    'employment_years': employment_years,
    'num_accounts': num_accounts,
    'region': region,
    'account_type': account_type
})
y = pd.Series(target, name='default')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print('Feature names:', feature_names)
print(f'Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}')
print(f'Class balance (full dataset):\n{y.value_counts(normalize=True).round(3)}')

## Part 1: n_estimators and OOB Score

In [ ]:
n_values = list(range(10, 310, 20))
oob_scores = []

for n_trees in n_values:
    rf = RandomForestClassifier(
        n_estimators=n_trees,
        oob_score=True,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    oob_scores.append(rf.oob_score_)

plt.figure(figsize=(9, 4))
plt.plot(n_values, oob_scores, marker='o', markersize=4)
plt.xlabel('n_estimators')
plt.ylabel('OOB Accuracy')
plt.title('OOB Score vs Number of Trees')
plt.tight_layout()
plt.show()

plateau_n = None
for i in range(1, len(oob_scores)):
    if abs(oob_scores[i] - oob_scores[i - 1]) < 0.001:
        plateau_n = n_values[i]
        break

print(f'OOB score plateaus (improvement < 0.001) at approximately n_estimators = {plateau_n}')

**Question 1:** The OOB score stops improving around n=120. Why does adding more trees beyond this point not hurt performance, even though it wastes compute? What does this tell you about the bias-variance tradeoff with `n_estimators`?

*(Write your answer here.)*

**Question 2:** The OOB score is described as approximately equivalent to leave-one-out CV. How is it computed internally? When would you still use proper cross-validation instead of relying on the OOB score?

*(Write your answer here.)*

## Part 2: max_features and Tree Correlation

In [ ]:
max_features_options = [1, 'sqrt', 'log2', None]
labels = ['1', 'sqrt', 'log2', 'None (all)']

print(f"{'max_features':<15} {'OOB Accuracy':<15}")
print('-' * 30)
for mf, label in zip(max_features_options, labels):
    rf = RandomForestClassifier(
        n_estimators=200,
        max_features=mf,
        oob_score=True,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    print(f"{label:<15} {rf.oob_score_:.4f}")

**Question 3:** `max_features=None` (all features at every split) produces the highest individual tree quality but typically lower ensemble performance. Why does using ALL features at every split undermine the ensemble?

*(Write your answer here.)*

**Question 4:** When would you lower `max_features` below `'sqrt'`? What signal would tell you the current setting is wrong?

*(Write your answer here.)*

## Part 3: MDI vs Permutation Importance

In [ ]:
# Add a customer_id column (sequential integers) — a textbook leaky-looking feature
X_train_aug = X_train.copy().reset_index(drop=True)
X_test_aug = X_test.copy().reset_index(drop=True)
X_train_aug['customer_id'] = np.arange(len(X_train_aug))
X_test_aug['customer_id'] = np.arange(len(X_train_aug), len(X_train_aug) + len(X_test_aug))

rf_full = RandomForestClassifier(
    n_estimators=200,
    oob_score=True,
    random_state=42,
    n_jobs=-1
)
rf_full.fit(X_train_aug, y_train)

# MDI importance
mdi_importances = pd.Series(
    rf_full.feature_importances_,
    index=X_train_aug.columns
).sort_values(ascending=False)

# Permutation importance (on test set)
perm_result = permutation_importance(
    rf_full, X_test_aug, y_test,
    n_repeats=15,
    random_state=42,
    n_jobs=-1
)
perm_importances = pd.Series(
    perm_result.importances_mean,
    index=X_train_aug.columns
).sort_values(ascending=False)

# Side-by-side table
importance_df = pd.DataFrame({
    'MDI': mdi_importances,
    'Permutation': perm_importances
}).sort_values('MDI', ascending=False)

print('Feature Importances — MDI vs Permutation')
print(importance_df.round(4).to_string())

**Question 5:** `customer_id` ranks high in MDI importance but near-zero in permutation importance. Explain the mechanism: why does a sequential integer create high impurity reduction in tree splits, even though it has no real predictive relationship with the target?

*(Write your answer here.)*

**Question 6:** You are producing a feature importance report for a stakeholder who will use it to decide which data to collect next. Which importance measure do you use, and why?

*(Write your answer here.)*

## Part 4: Partial Dependence Plots

In [ ]:
# Use the base model (without customer_id) for interpretable PDPs
rf_pdp = RandomForestClassifier(
    n_estimators=200,
    oob_score=True,
    random_state=42,
    n_jobs=-1
)
rf_pdp.fit(X_train, y_train)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
PartialDependenceDisplay.from_estimator(
    rf_pdp,
    X_train,
    features=['income', 'credit_score'],
    kind='average',
    ax=ax
)
plt.suptitle('Partial Dependence Plots — income and credit_score', y=1.02)
plt.tight_layout()
plt.show()

**Question 7:** A PDP shows the expected model output as a function of one feature, with all other features "averaged out." For a feature that is highly correlated with another (e.g., `income` and `debt_ratio`), what limitation does the PDP have? What combinations of feature values might it be averaging over that would never appear in real data?

*(Write your answer here.)*

**Question 8:** `kind='individual'` (ICE plots) shows one line per observation. When would ICE plots reveal something that the average PDP hides? Give a concrete example involving a feature that affects different subgroups differently.

*(Write your answer here.)*

## Summary

Check your understanding with these final questions. Each should be answerable in one sentence.

1. Why does the OOB error become a reliable performance estimate only once you have enough trees, and what does it mean for variance when you add more trees beyond the plateau?

2. In one sentence, explain why reducing `max_features` decreases tree correlation and why that generally improves ensemble generalisation.

3. In one sentence, state the key difference between MDI and permutation importance that makes MDI unreliable for high-cardinality or continuous features.

4. A PDP for `credit_score` shows a flat line between 600 and 750. Before concluding that credit score does not matter in this range, what alternative explanation should you check?